<a href="https://colab.research.google.com/github/tebitaan13-hub/curso-programacion-cientifica/blob/main/taller_evaluativo_02_funciones_y_poo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Taller evaluativo 02 — Funciones y POO


Ejercicio 1 — Calibrar una lectura

Se define una función reutilizable que aplica la recta de calibración. Si no se indican parámetros, la lectura queda sin cambios.

In [10]:
def calibrar_lectura(valor, pendiente=1.0, intercepto=0.0):
    """Aplica la recta de calibracion a una lectura."""
    return valor * pendiente + intercepto


sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto=2.0)

print(sin_calibrar)
print(calibrada)
print(con_intercepto)

10.0
11.5
12.0


Ejercicio 2 — ¿Es utilizable la lectura?

Se guardan las lecturas crudas de SBC-A y se define una función que identifica las lecturas válidas: no deben ser el centinela `-999.0` y deben estar entre 0 y 500 inclusive.

In [11]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


def es_lectura_valida(valor):
    """Indica si una lectura esta dentro del rango utilizable."""
    return valor != -999.0 and 0.0 <= valor <= 500.0


print(es_lectura_valida(25.0))
print(es_lectura_valida(-999.0))
print(es_lectura_valida(600.0))
print(es_lectura_valida(0.0))

True
False
False
True


Ejercicio 3 — Promedio de las lecturas útiles

La función recorre la lista recibida, suma únicamente las lecturas válidas mediante `es_lectura_valida` y devuelve el promedio. Si no hay lecturas válidas, devuelve `None`.

In [12]:
def promedio_valido(lecturas):
    """Calcula el promedio de las lecturas validas o devuelve None."""
    total = 0.0
    cantidad = 0
    for valor in lecturas:
        if es_lectura_valida(valor):
            total += valor
            cantidad += 1
    if cantidad == 0:
        return None
    return total / cantidad


promedio_sbc_a = promedio_valido(lecturas_sbc_a)
print(promedio_sbc_a)
print(promedio_valido([-999.0, -999.0]))

27.0
None


Ejercicio 4 — Calibrar la serie con map, filter y lambda

Se eliminan las lecturas no utilizables con `filter`, se calibran las restantes con `map` y una función `lambda`, y se cuentan las concentraciones calibradas que superan 35.4 µg/m³. En este ejercicio no se usan bucles ni comprensiones de listas.

In [13]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

lecturas_utiles = list(filter(es_lectura_valida, lecturas_sbc_a))
lecturas_calibradas = list(
    map(lambda valor: calibrar_lectura(valor, pendiente_sbc_a, intercepto_sbc_a),
        lecturas_utiles)
)
excedencias = len(list(filter(lambda valor: valor > 35.4, lecturas_calibradas)))

print(lecturas_calibradas)
print(excedencias)

[11.5, 29.5, 47.5, 9.1, 61.9]
2


Ejercicio 5 — La clase Lectura

La clase representa una lectura individual. Guarda su valor, unidad y código de sensor; puede convertir a µg/m³ e identificar el valor centinela. En este ejercicio no valida ni rechaza valores.

In [14]:
class Lectura:
    """Representa una lectura de un sensor de calidad del aire."""

    def __init__(self, valor, unidad="ug/m3", codigo_sensor="sin asignar"):
        """Guarda el valor, la unidad y el codigo del sensor."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve el valor convertido a microgramos por metro cubico."""
        if self.unidad == "mg/m3":
            return self.valor * 1000
        return self.valor

    def es_dato_faltante(self):
        """Indica si el valor corresponde al centinela de dato faltante."""
        return self.valor == -999.0


lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print(lectura_campo.a_microgramos())
print(lectura_faltante.es_dato_faltante())
print(lectura_campo.es_dato_faltante())
print(lectura_normal.unidad, lectura_normal.codigo_sensor)

25.2
True
False
ug/m3 sin asignar


Ejercicio 6 — La clase Sensor: validación y encapsulamiento

La clase valida el código y la pendiente antes de guardar atributos. La pendiente y el intercepto son internos (`_pendiente`, `_intercepto`), y el método `calibrar` reutiliza `calibrar_lectura`.

In [15]:
class Sensor:
    """Representa un sensor con parametros de calibracion."""

    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        """Valida y guarda los datos iniciales del sensor."""
        if codigo == "":
            raise ValueError("El codigo del sensor no puede estar vacio")
        if pendiente <= 0:
            raise ValueError("La pendiente debe ser mayor que cero")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Devuelve una lectura calibrada con los parametros del sensor."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        """Devuelve la representacion tecnica del sensor."""
        return f"Sensor(codigo={self.codigo!r}, pendiente={self._pendiente}, intercepto={self._intercepto})"

    def __str__(self):
        """Devuelve una descripcion legible del sensor."""
        return f"Sensor {self.codigo}: pendiente={self._pendiente}, intercepto={self._intercepto}"


sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print(sensor_a)
print([sensor_a, sensor_b])
print(sensor_a.calibrar(10.0))
print(sensor_b.calibrar(10.0))

# Pruebas de validacion opcionales: descomente una linea a la vez para comprobar el ValueError.
# Sensor("")
# Sensor("X", 0)

Sensor SBC-A: pendiente=1.2, intercepto=-0.5
[Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
11.5
10.0


 Ejercicio 7 — La clase Estacion: objetos que contienen objetos

La estación contiene una lista interna de objetos `Lectura`. El método `valores()` obtiene las concentraciones en µg/m³ llamando al método `a_microgramos()` de cada lectura.

In [16]:
class Estacion:
    """Representa una estacion que contiene lecturas."""

    def __init__(self, codigo, sector):
        """Inicializa la estacion con una lista de lecturas vacia."""
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega un objeto Lectura a la estacion."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve las lecturas convertidas a microgramos por metro cubico."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __repr__(self):
        """Devuelve la representacion tecnica de la estacion."""
        return f"Estacion({self.codigo!r}, {self.sector!r}, lecturas={len(self._lecturas)})"


datos_campana = [
    (10.0, "ug/m3"), (-999.0, "ug/m3"), (0.025, "mg/m3"), (40.0, "ug/m3"),
    (-999.0, "ug/m3"), (8.0, "ug/m3"), (52.0, "ug/m3"), (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")
for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, "SBC-A"))

print(estacion_oriente)
print(estacion_oriente.valores())

Estacion('EST-04', 'Oriente', lecturas=8)
[10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


 Ejercicio 8 — Integración: el informe de la estación

Se define la función de clasificación y la clase `InformeEstacion`, que obtiene los valores de la estación, filtra lecturas no válidas, las calibra con el sensor y calcula el promedio reutilizando `promedio_valido`. El informe muestra el promedio redondeado y su categoría.

In [17]:
def categoria_pm25(concentracion):
    """Devuelve la categoria oficial correspondiente a una concentracion PM2.5."""
    if concentracion <= 12.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"


class InformeEstacion:
    """Genera un informe de calidad del aire para una estacion y un sensor."""

    def __init__(self, estacion, sensor):
        """Guarda la estacion y el sensor asociados al informe."""
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Calcula el promedio de las lecturas validas despues de calibrarlas."""
        valores_utiles = filter(es_lectura_valida, self._estacion.valores())
        valores_calibrados = list(map(self._sensor.calibrar, valores_utiles))
        return promedio_valido(valores_calibrados)

    def categoria(self):
        """Devuelve la categoria del promedio calibrado de la estacion."""
        promedio = self.promedio_calibrado()
        if promedio is None:
            return "Sin datos validos"
        return categoria_pm25(promedio)

    def __str__(self):
        """Devuelve el informe legible de la estacion."""
        promedio = self.promedio_calibrado()
        if promedio is None:
            return f"{self._estacion.codigo}: sin datos validos"
        return f"{self._estacion.codigo}: {round(promedio, 1)} ug/m3 - {self.categoria()}"


informe = InformeEstacion(estacion_oriente, sensor_a)
print(informe)
print(informe.promedio_calibrado())

EST-04: 31.9 ug/m3 - Moderada
31.9


Celda de verificación final



In [18]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
